# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables


import os
from dotenv import load_dotenv

load_dotenv("config.env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL




### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import chromadb
from chromadb.utils import embedding_functions

# Reconnect to the existing collection
embedding_fn = embedding_functions.DefaultEmbeddingFunction()
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

def retrieve_game(query: str) -> list:
 
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    
    games = []
    for metadata in results["metadatas"][0]:
        games.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description")
        })
    
    return games



Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

import json
from pydantic import BaseModel
from openai import OpenAI


class EvaluationReport(BaseModel):
    useful: bool
    description: str

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL
)

def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationReport:
  
    docs_text = json.dumps(retrieved_docs, indent=2)

    prompt = f"""Your task is to evaluate if the documents are enough to respond to the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

Question: {question}

Retrieved Documents:
{docs_text}

Respond in JSON format with exactly these fields:
- "useful": true or false, whether the documents are sufficient to answer the question
- "description": a detailed explanation of your evaluation

Respond with JSON only, no extra text."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    result = json.loads(response.choices[0].message.content)
    return EvaluationReport(
        useful=result["useful"],
        description=result["description"]
    )

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def game_web_search(question: str) -> str:

    response = tavily_client.search(question)
    
    results = []
    for result in response["results"]:
        results.append({
            "title": result["title"],
            "content": result["content"],
            "url": result["url"]
        })
    
    return results

### Agent

In [7]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

from lib.tooling import tool

@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds most relevant games in the vector DB for a given query."""
    results = collection.query(query_texts=[query], n_results=3)
    games = []
    for metadata in results["metadatas"][0]:
        games.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description")
        })
    return games

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    """Analyzes whether the retrieved documents are sufficient to answer the user's question."""
    import json
    
    # Handle all possible input types safely
    if isinstance(retrieved_docs, list):
        docs = retrieved_docs
    elif isinstance(retrieved_docs, str):
        try:
            docs = json.loads(retrieved_docs)
        except json.JSONDecodeError:
            docs = [{"content": retrieved_docs}]
    else:
        docs = [{"content": str(retrieved_docs)}]
    
    docs_text = json.dumps(docs, indent=2)
    prompt = f"""Evaluate if the documents are enough to respond to the query.
Question: {question}
Retrieved Documents:
{docs_text}

You MUST respond with ONLY a valid JSON object, no markdown, no backticks, no explanation.
Example: {{"useful": true, "description": "The documents contain relevant information."}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    
    result = json.loads(response.choices[0].message.content)
    return {"useful": result["useful"], "description": result["description"]}

@tool
def game_web_search(question: str) -> list:
    """Searches the web for video game information using Tavily when internal knowledge is insufficient."""
    response = tavily_client.search(question)
    results = []
    for result in response["results"]:
        results.append({
            "title": result["title"],
            "content": result["content"],
            "url": result["url"]
        })
    return results

# creating agent
agent = Agent(
    # groq model i've been using
    model_name="gpt-4o-mini",
    instructions="""You are UdaPlay, an expert gaming assistant for a gaming analytics company.
    
Your task is to answer questions about video games, which includes:
- game titles and their details
- release dates and platforms  
- game descriptions and genres
- info about the publisher

Follow this workflow for every question:
1. ALWAYS start by using retrieve_game to search the internal database
2. ALWAYS use evaluate_retrieval to assess whether the retrieved results are useful
3. If evaluate_retrieval says the results are NOT useful, use game_web_search to find the answer online
4. Combine information from all sources to give a clear, structured answer
5. ALWAYS cite where the information came from (internal database or web search)
    - For database results: include the filename e.g. [Source: local_db/Gran_Turismo.json]
    - For web results: include the full URL e.g. [Source: https://www.example.com/article]
    - NEVER give an answer without at least one source citation

Be concise, accurate and as helpful as possible.""",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3
)

print("agent created successfully")

agent created successfully


In [8]:
test_docs = [{"Name": "Pokemon Gold", "Platform": "Game Boy Color", "YearOfRelease": 1999, "Description": "RPG game"}]
result = evaluate_retrieval("When was Pokemon Gold and Silver released?", test_docs)
print(result)

{'useful': True, 'description': 'The document provides the release year of Pokemon Gold, and since Pokemon Silver was released the same year, it answers the query.'}


In [9]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?


# each separate question gets its own session ID
questions = [
    ("When was Pokémon Gold and Silver released?", "pokemon"),
    ("Which one was the first 3D platformer Mario game?", "mario"),
    ("Was Mortal Kombat X released for Playstation 5?", "mortal_kombat")
]

for question, session_id in questions:
    print(f"\n{'='*50}")
    print(f"Question: {question}")
    print('='*50)
    
    run = agent.invoke(question, session_id=session_id)
    final_state = run.get_final_state()
    messages = final_state["messages"]
    
    # show tool calls and results
    print("\n--- Agent Decision Trace ---")
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"   Tool Called: {tc.function.name}")
                print(f"   Input: {tc.function.arguments}")
        if hasattr(msg, 'role') and msg.role == 'tool':
            print(f"   Result summary: {msg.content[:200]}...")
    
    # final answer
    print("\n--- Final Answer ---")
    last_message = messages[-1]
    print(last_message.content)

# multi-turn demo
print("\n\n" + "="*50)
print("MULTI-TURN CONVERSATION DEMO (same session_id)")
print("="*50)

followup_questions = [
    "What platforms was Gran Turismo released on?",
    "Who published it?",
]

for question in followup_questions:
    print(f"\nQuestion: {question}")
    print("-"*50)
    run = agent.invoke(question, session_id="gran_turismo_chat")
    final_state = run.get_final_state()
    messages = final_state["messages"]
    
    print("\n--- Agent Decision Trace ---")
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"   Tool Called: {tc.function.name}")
                print(f"   Input: {tc.function.arguments}")
        if hasattr(msg, 'role') and msg.role == 'tool':
            print(f"   Result summary: {msg.content[:200]}...")
    
    print("\n--- Final Answer ---")
    last_message = messages[-1]
    print(last_message.content)

# web search fallback demo
print("\n\n" + "="*50)
print("WEB SEARCH FALLBACK DEMO")
print("="*50)

web_question = "What is the latest Call of Duty game released in 2024?"
print(f"\nQuestion: {web_question}")
print("-"*50)

run = agent.invoke(web_question, session_id="web_search_demo")
final_state = run.get_final_state()
messages = final_state["messages"]

print("\n--- Agent Decision Trace ---")
for msg in messages:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"   Tool Called: {tc.function.name}")
            print(f"   Input: {tc.function.arguments}")
    if hasattr(msg, 'role') and msg.role == 'tool':
        print(f"   Result summary: {msg.content[:200]}...")

print("\n--- Final Answer ---")
last_message = messages[-1]
print(last_message.content)



Question: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor


sh: 1: blkid: not found
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- Agent Decision Trace ---
   Tool Called: retrieve_game
   Input: {"query":"Pokémon Gold and Silver release date"}
   Result summary: "[{'Name': 'Pok\u00e9mon Gold and Silver', 'Platform': 'Game Boy Color', 'YearOfRelease': 1999, 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay...
   Tool Called: evaluate_retrieval
   Input: {"question":"When was Pokémon Gold and Silver released?","retrieved_docs":"[{'Name': 'Pokémon Gold and Silver', 'Platform': 'Game Boy Color', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Name': 'Pokémon Ruby and Sapphire', 'Platform': 'Game Boy Advance', 'YearOfRelease': 2002, 'Description': 'Third-

### (Optional) Advanced

In [10]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes